In [43]:
from google.colab import drive
drive.mount('/content/drive')
%ls drive/MyDrive/PhishingModels
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
test.gdoc
Wed May 13 01:04:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|

In [11]:
#IMPORTS

import joblib as jl
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [12]:
#DATASET

df = pd.read_csv("/content/drive/MyDrive/SevenPhishingEmails/scikit_cleaned.csv")

texts = (
    df['sender'].fillna('') + ' ' +
    df['receiver'].fillna('') + ' ' +
    df['date'].fillna('') + ' ' +
    df['subject'].fillna('') + ' ' +
    df['body'].fillna('')
)

y = df['label']

In [13]:
#60-20-20 SPLIT

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    texts, y, test_size=0.4, random_state=42
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.5, random_state=42
)

In [14]:
#VECTORIZER

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

In [15]:
#MODEL

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [23]:
#VALIDATION

val_pred = model.predict(X_val)

val_acc = accuracy_score(y_val, val_pred)

print("Validation Accuracy:", val_acc)
print(confusion_matrix(y_val, val_pred))
print(classification_report(y_val, val_pred))

Validation Accuracy: 0.9745456888774326
[[7768   69]
 [ 326 7355]]
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      7837
           1       0.99      0.96      0.97      7681

    accuracy                           0.97     15518
   macro avg       0.98      0.97      0.97     15518
weighted avg       0.98      0.97      0.97     15518



In [51]:
#FINAL TEST AND DOWNLOAD

def final_test():
    test_pred = model.predict(X_test)

    test_acc = accuracy_score(y_test, test_pred)

    print("\nFINAL TEST RESULTS")
    print(confusion_matrix(y_test, test_pred))
    print(classification_report(y_test, test_pred))
    print("Test Accuracy:", test_acc)

    metadata = {
    "validation_accuracy": val_acc,
    "test_accuracy": test_acc
    }

    jl.dump(model, "/content/drive/MyDrive/PhishingModels/scikit_model.pkl")
    jl.dump(vectorizer, "/content/drive/MyDrive/PhishingModels/scikit_vectorizer.pkl")
    jl.dump(metadata, "/content/drive/MyDrive/PhishingModels/scikit_metadata.pkl")

In [52]:
#Commented this out for myself so I don't run it accidentally like an idiot.

final_test()


FINAL TEST RESULTS
[[7794   58]
 [ 303 7363]]
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      7852
           1       0.99      0.96      0.98      7666

    accuracy                           0.98     15518
   macro avg       0.98      0.98      0.98     15518
weighted avg       0.98      0.98      0.98     15518

Test Accuracy: 0.9767366928727929
